# EV Policy Assistant

## Stage 1: environment checks

Setup only; PDF ingestion starts in a later stage. Adapted from the course repository at commit `33c2faa22450cde16ead9071f7ce7ecc78ca592a`: Lab 4 cells 6–8, 38, 65, 77, 106 and 150; Exercise 2 cell 4. AI assistance adapted these setup checks; no policy-answering pipeline is implemented here.

Run from the project folder with the project’s Python 3.12 environment. Put your Groq key in the local `.env` file. Notebook outputs should be cleared before committing.

In [ ]:
import os
import sys
import math
from importlib.metadata import version
from dotenv import load_dotenv
import gradio as gr
from langchain.chat_models import init_chat_model
from langchain.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(override=True)

assert sys.version_info[:2] == (3, 12), "Select the project Python 3.12 kernel."
print("Python:", sys.version.split()[0])
for package in ["langchain", "langchain-chroma", "langchain-ollama", "langchain-groq", "gradio", "pypdf", "unstructured"]:
    print(package, version(package))

### Local embeddings

Ollama must be running with `nomic-embed-text` available. This checks one query; it does not build an index.

In [ ]:
embeddings_model = OllamaEmbeddings(model="nomic-embed-text")
query_embedding = embeddings_model.embed_query("EV policy setup check")
assert len(query_embedding) > 0
assert all(math.isfinite(value) for value in query_embedding)
print("Embedding dimensions:", len(query_embedding))

### Groq connection

This sends a small test prompt to Groq and uses the account’s API allowance. A missing key stops the check; it is not a successful connection test.

In [ ]:
if not os.environ.get("GROQ_API_KEY"):
    raise ValueError("Add GROQ_API_KEY to the local .env file and rerun the setup cells.")

model_name = "openai/gpt-oss-120b"
llm = init_chat_model(model_name, model_provider="groq", temperature=0, max_tokens=256, timeout=30, max_retries=0)
response = llm.invoke("Reply with only OK.")
assert response.content, "Groq returned an empty response."
print(response.content)